In [1]:
import sys
print(sys.executable)

C:\Users\srish\anaconda3\envs\maddpg\python.exe


In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

PyTorch: 2.14.0+cpu
CUDA: False


In [3]:
import gymnasium as gym

env = gym.make("CartPole-v1")
print("Gymnasium working!")
env.close()

ModuleNotFoundError: No module named 'gymnasium'

In [4]:
!python a3c.py

Traceback (most recent call last):
  File "C:\Users\srish\a3c.py", line 5, in <module>
    import gymnasium as gym
ModuleNotFoundError: No module named 'gymnasium'


In [5]:
python -m pip install "gymnasium[classic-control]"

SyntaxError: invalid syntax (4161966305.py, line 1)

In [6]:
!pip install "gymnasium[classic-control]"

  Using cached gymnasium-1.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached farama_notifications-0.0.6-py3-none-any.whl.metadata (729 bytes)
Using cached gymnasium-1.3.0-py3-none-any.whl (953 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached farama_notifications-0.0.6-py3-none-any.whl (2.9 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.7 MB 2.1 MB/s eta 0:00:05
   ---- ----------------------------------- 1.0/9.7 MB 2.0 MB/s eta 0:00:05
   ----- ---------------------------------- 1.3/9.7 MB 1.9 MB/s eta 0:00:05
   ------- -------------------------------- 1.8/9.7 MB 1.9 MB/s eta 0:00:05
   --------- ------------------------------ 2.4/9.7 MB 2.0 MB/s eta 0:00:04
   ---------- ----------------------------- 2.6/9.7 MB 2.0 MB/s eta 0:00:04
   -------------

In [7]:
import gymnasium as gym

env = gym.make("CartPole-v1")
print("Gymnasium working!")
env.close()

Gymnasium working!


In [8]:
!python a3c.py

ep=10 worker=1 reward=17.0
ep=20 worker=2 reward=19.0
ep=30 worker=2 reward=12.0
ep=40 worker=2 reward=15.0
ep=50 worker=0 reward=10.0
ep=60 worker=3 reward=12.0
ep=70 worker=0 reward=13.0
ep=80 worker=3 reward=12.0
ep=90 worker=0 reward=14.0
ep=100 worker=2 reward=23.0
ep=110 worker=3 reward=16.0
ep=120 worker=0 reward=25.0
ep=130 worker=0 reward=20.0
ep=140 worker=2 reward=10.0
ep=150 worker=2 reward=9.0
ep=160 worker=3 reward=23.0
ep=170 worker=2 reward=39.0
ep=180 worker=2 reward=16.0
ep=190 worker=2 reward=11.0
ep=200 worker=1 reward=12.0


In [9]:
%%writefile a3c.py

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.multiprocessing as mp
import gymnasium as gym

GAMMA = 0.99
N_STEPS = 5
LR = 1e-3
WORKERS = 4
MAX_EP = 200


class Net(nn.Module):

    def __init__(self, s, a):
        super().__init__()

        self.body = nn.Sequential(
            nn.Linear(s, 128),
            nn.ReLU()
        )

        self.actor = nn.Linear(128, a)
        self.critic = nn.Linear(128, 1)

    def forward(self, x):
        h = self.body(x)
        return self.actor(h), self.critic(h)

    def act(self, state):

        state = torch.tensor(
            state,
            dtype=torch.float32
        ).unsqueeze(0)

        logits, v = self.forward(state)

        dist = torch.distributions.Categorical(
            F.softmax(logits, dim=-1)
        )

        a = dist.sample()

        return a.item(), dist.log_prob(a), v


def worker(wid, gmodel, opt, counter, q):

    env = gym.make("CartPole-v1")

    local = Net(
        env.observation_space.shape[0],
        env.action_space.n
    )

    local.load_state_dict(
        gmodel.state_dict()
    )

    while counter.value < MAX_EP:

        s, _ = env.reset()

        done = False
        ep_r = 0

        logp = []
        vals = []
        rews = []

        while not done:

            a, lp, v = local.act(s)

            s2, r, term, trunc, _ = env.step(a)

            done = term or trunc

            logp.append(lp)
            vals.append(v)
            rews.append(r)

            ep_r += r
            s = s2

            if len(rews) == N_STEPS or done:

                R = 0.0 if done else local.act(s)[2].item()

                returns = []

                for r_ in reversed(rews):
                    R = r_ + GAMMA * R
                    returns.insert(0, R)

                returns = torch.tensor(
                    returns,
                    dtype=torch.float32
                )

                values = torch.cat(vals).squeeze(-1)

                adv = returns - values

                loss = (
                    -(torch.stack(logp) * adv.detach()).mean()
                    + 0.5 * adv.pow(2).mean()
                )

                opt.zero_grad()

                loss.backward()

                for lp_, gp_ in zip(
                    local.parameters(),
                    gmodel.parameters()
                ):
                    if lp_.grad is not None:
                        gp_._grad = lp_.grad

                opt.step()

                local.load_state_dict(
                    gmodel.state_dict()
                )

                logp = []
                vals = []
                rews = []

        with counter.get_lock():

            counter.value += 1

            ep = counter.value

        q.put(
            (wid, ep, ep_r)
        )

    env.close()


def main():

    env = gym.make("CartPole-v1")

    gmodel = Net(
        env.observation_space.shape[0],
        env.action_space.n
    )

    gmodel.share_memory()

    opt = torch.optim.Adam(
        gmodel.parameters(),
        lr=LR
    )

    counter = mp.Value("i", 0)

    q = mp.Queue()

    procs = [
        mp.Process(
            target=worker,
            args=(i, gmodel, opt, counter, q)
        )
        for i in range(WORKERS)
    ]

    for p in procs:
        p.start()

    for i in range(MAX_EP):

        wid, ep, r = q.get()

        if ep % 10 == 0:

            print(
                f"ep={ep} worker={wid} reward={r}"
            )

    for p in procs:
        p.join()

    env.close()


if __name__ == "__main__":

    try:

        mp.set_start_method(
            "spawn",
            force=True
        )

    except RuntimeError:
        pass

    main()

            if len(rews) == N_STEPS or done:

            R = 0.0 if done else local.act(s)[2].item()

            returns = []

            for r_ in reversed(rews):
                R = r_ + GAMMA * R
                returns.insert(0, R)

            returns = torch.tensor(
                returns,
                dtype=torch.float32
            )

            adv = returns - torch.cat(vals).squeeze(-1)

            loss = (
                -(torch.stack(logp) * adv.detach()).mean()
                + 0.5 * adv.pow(2).mean()
            )

            opt.zero_grad()
            loss.backward()

            for lp_, gp_ in zip(
                local.parameters(),
                gmodel.parameters()
            ):
                gp_._grad = lp_.grad

            opt.step()

            local.load_state_dict(
                gmodel.state_dict()
            )

            logp = []
            vals = []
            rews = []

    with counter.get_lock():
        counter.value += 1
        ep = counter.value

    q.put((wid, ep, ep_r))

    env.close()


def main():

    env = gym.make("CartPole-v1")

    gmodel = Net(
        env.observation_space.shape[0],
        env.action_space.n
    )

    gmodel.share_memory()

    opt = torch.optim.Adam(
        gmodel.parameters(),
        lr=LR
    )

    counter = mp.Value("i", 0)
    q = mp.Queue()

    procs = [
        mp.Process(
            target=worker,
            args=(i, gmodel, opt, counter, q)
        )
        for i in range(WORKERS)
    ]

    for p in procs:
        p.start()

    for i in range(MAX_EP):

        wid, ep, r = q.get()

        if ep % 10 == 0:
            print(
                f"ep={ep} worker={wid} reward={r}"
            )

    for p in procs:
        p.join()

    env.close()


if __name__ == "__main__":

    try:
        mp.set_start_method(
            "spawn",
            force=True
        )
    except RuntimeError:
        pass

    main()

Overwriting a3c.py


In [10]:
!python a3c.py

  File "C:\Users\srish\a3c.py", line 208
    if len(rews) == N_STEPS or done:
IndentationError: unexpected indent
